# LLM13: Sparse Attention & PagedAttention

## Lab Overview

Standard attention has $O(s^2)$ memory and compute cost, making long-context inference expensive. This lab explores **Sparse Attention** — selecting a subset of keys/values instead of attending to all — and **PagedAttention**, which manages KV-Cache memory efficiently via paging.

> **Quick term:** **PagedAttention** stores KV-cache in fixed-size **blocks** (pages) that can be allocated non-contiguously—like OS virtual memory—so a serving engine can batch many sequences without huge contiguous buffers.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Implement static sparse patterns (sliding window, attention sinks), visualize attention sparsity, and understand dynamic selection methods.

1. **Explain the KV-Cache Bottleneck**: Why full attention becomes impractical for long sequences.
2. **Implement Sliding Window Attention**: A static sparse pattern that limits each token to attend to a local window.
3. **Implement Attention Sinks (StreamingLLM)**: Keep initial tokens + sliding window for stable streaming inference.
4. **Understand Dynamic Sparse Patterns**: MInference (vertical-slash, block-sparse) and Quest (page-level scoring).
5. **Understand PagedAttention**: Block-based KV memory management for efficient serving.

---


## 1. Environment Setup

In [1]:
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.6.0+gitdbfe118
GPU: AMD Radeon Graphics
GPU Memory: 12.4 GB


## 2. Attention Sparsity Observation

### Recap: Full Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The attention matrix $A \in \mathbb{R}^{s \times s}$ is **dense** in the sense that every query *may* attend to every key. **Full attention costs $O(s^2)$** in compute and memory, which becomes the bottleneck for long contexts. **Sparse attention** methods address this by **restricting which key positions each query uses** (e.g. sliding window, block-sparse, sinks) so you do not materialize or use the full $s \times s$ score matrix—motivated primarily by **scaling**, not by softmax values being "mostly zero" (trained models often have **peaked** rows, but quadratic cost remains). Formally, one selects a subset of keys $K_{s'} \subseteq K$ per query (or a fixed pattern):

$$\hat{A} = \text{softmax}\left(\frac{Q K_{s'}^T}{\sqrt{d_k}}\right) V_{s'}$$

Let's visualize this sparsity.

In [2]:
def compute_attention_weights(seq_len, d_model=64, num_heads=4):
    """Generate a synthetic causal attention weight matrix."""
    head_dim = d_model // num_heads
    Q = torch.randn(1, num_heads, seq_len, head_dim)
    K = torch.randn(1, num_heads, seq_len, head_dim)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(head_dim)

    # Apply causal mask
    causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1)
    return attn_weights  # (1, num_heads, seq_len, seq_len)


seq_len = 64
attn = compute_attention_weights(seq_len)

# Analyze sparsity: what fraction of weights are < threshold?
print("=== Attention Weight Sparsity ===")
for threshold in [0.01, 0.005, 0.001]:
    # Only look at non-masked positions (lower triangle)
    mask = ~torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    valid_weights = attn[0, 0][mask]
    sparse_frac = (valid_weights < threshold).float().mean()
    print(f"  Threshold {threshold}: {sparse_frac*100:.1f}% of weights are below it")

print(f"\nWith seq_len={seq_len}, full attention computes {seq_len*(seq_len+1)//2:,} scores (causal).")
print("Note: Random Q,K make softmax diffuse; real trained models often show peaked attention. Sparse methods mainly cut the O(s²) cost, not only 'small weights'.")

=== Attention Weight Sparsity ===
  Threshold 0.01: 36.0% of weights are below it
  Threshold 0.005: 17.1% of weights are below it
  Threshold 0.001: 1.7% of weights are below it

With seq_len=64, full attention computes 2,080 scores (causal).
Note: Random Q,K make softmax diffuse; real trained models often show peaked attention. Sparse methods mainly cut the O(s²) cost, not only 'small weights'.


In [3]:
# Visualize attention as text heatmap (head 0)
attn_head0 = attn[0, 0].numpy()  # (seq_len, seq_len)

# Show a 16x16 sub-block for readability
block = attn_head0[:16, :16]
print("=== Attention Weights (Head 0, first 16 tokens) ===")
print("Row=query, Col=key. Brighter = higher weight.\n")

symbols = " ░▒▓█"
header = "     " + "".join(f"{j:>3d}" for j in range(16))
print(header)
for i in range(16):
    row = f"q{i:>2d}: "
    for j in range(16):
        w = block[i, j]
        idx = min(int(w * len(symbols)), len(symbols) - 1)
        row += f"  {symbols[idx]}"
    print(row)

print("\nLegend: ' '=0, ░=low, ▒=medium, ▓=high, █=very high")
print("Notice: attention concentrates on nearby tokens and the very first token.")

=== Attention Weights (Head 0, first 16 tokens) ===
Row=query, Col=key. Brighter = higher weight.

       0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15
q 0:   █                                             
q 1:   ▓  ░                                          
q 2:   ▒     ░                                       
q 3:      ▒  ░                                       
q 4:   ▒        ░                                    
q 5:         ▒     ░                                 
q 6:   ░                                             
q 7:                     ░                           
q 8:         ░  ░  ░                                 
q 9:   ▒                                             
q10:   ░                                             
q11:                     ░        ░                  
q12:      ░                       ░                  
q13:                                                 
q14:                                                 
q15:                                 

## 3. Sliding Window Attention

The simplest sparse attention: each query only attends to the last $w$ keys (local window).

**Complexity**: $O(s \cdot w)$ instead of $O(s^2)$ — linear in $s$ when $w$ is fixed.

Used in **Mistral**, **Mixtral**, and other modern models.

In [4]:
def sliding_window_attention(Q, K, V, window_size):
    """
    Sliding window attention: each query attends to at most `window_size` preceding keys.

    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        window_size: number of preceding tokens to attend to
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    B, H, S, D = Q.shape
    scale = 1.0 / math.sqrt(D)

    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale  # (B, H, S, S)

    # Create sliding window mask: position i can attend to [max(0, i-w+1), i]
    positions = torch.arange(S)
    # mask[i, j] = True if j should be MASKED OUT
    mask = (positions.unsqueeze(0) - positions.unsqueeze(1)) > 0  # causal
    mask = mask | ((positions.unsqueeze(1) - positions.unsqueeze(0)) >= window_size)  # window

    scores.masked_fill_(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights


def full_causal_attention(Q, K, V):
    """Standard causal attention for comparison."""
    S = Q.size(2)
    scale = 1.0 / math.sqrt(Q.size(-1))
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
    causal_mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
    scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights


# Compare
B, H, S, D = 1, 4, 32, 16
Q = torch.randn(B, H, S, D)
K = torch.randn(B, H, S, D)
V = torch.randn(B, H, S, D)

out_full, w_full = full_causal_attention(Q, K, V)
out_sw, w_sw = sliding_window_attention(Q, K, V, window_size=8)

# Sparsity comparison
full_nonzero = (w_full[0, 0] > 1e-6).sum().item()
sw_nonzero = (w_sw[0, 0] > 1e-6).sum().item()

print("=== Sliding Window vs Full Attention (Head 0) ===")
print(f"Full attention: {full_nonzero} non-zero entries ({full_nonzero/(S*S)*100:.1f}% of matrix)")
print(f"Window (w=8):   {sw_nonzero} non-zero entries ({sw_nonzero/(S*S)*100:.1f}% of matrix)")
print(f"Memory savings:  {(1 - sw_nonzero/full_nonzero)*100:.1f}% fewer attention scores")
print(f"\nOutput difference (MSE): {F.mse_loss(out_full, out_sw).item():.6f}")

=== Sliding Window vs Full Attention (Head 0) ===
Full attention: 528 non-zero entries (51.6% of matrix)
Window (w=8):   228 non-zero entries (22.3% of matrix)
Memory savings:  56.8% fewer attention scores

Output difference (MSE): 0.091904


## 4. Attention Sinks — StreamingLLM

**Observation** (Xiao et al., 2023): The first few tokens ("attention sinks") consistently receive high attention weights, regardless of their semantic importance. Dropping them destabilizes generation.

**StreamingLLM pattern**: Keep $k_{\text{sink}}$ initial tokens + a sliding window of recent tokens.

$$\text{Keep tokens}: \{0, 1, \ldots, k_{\text{sink}}-1\} \cup \{i - w + 1, \ldots, i\}$$

This enables streaming inference on arbitrarily long text with **fixed** KV-Cache size.

In [5]:
def sink_window_attention(Q, K, V, window_size, n_sink=4):
    """
    StreamingLLM-style attention: attend to first n_sink tokens + sliding window.

    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        window_size: local window size
        n_sink: number of initial tokens to always attend to
    Returns:
        output, attention_weights
    """
    B, H, S, D = Q.shape
    scale = 1.0 / math.sqrt(D)

    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale

    positions = torch.arange(S)
    # Causal mask
    causal = positions.unsqueeze(0) > positions.unsqueeze(1)

    # Window mask: j < i - window_size + 1
    window = (positions.unsqueeze(1) - positions.unsqueeze(0)) >= window_size

    # Sink exception: always allow attending to first n_sink positions
    is_sink = positions.unsqueeze(0) < n_sink  # (1, S) -> broadcast

    # Final mask: masked out if causal AND (outside window AND not a sink)
    mask = causal | (window & ~is_sink)

    scores.masked_fill_(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights


# Compare all three patterns
S = 32
Q = torch.randn(1, 1, S, 16)
K = torch.randn(1, 1, S, 16)
V = torch.randn(1, 1, S, 16)
w = 8
n_sink = 4

out_full, w_full = full_causal_attention(Q, K, V)
out_sw, w_sw = sliding_window_attention(Q, K, V, window_size=w)
out_sink, w_sink = sink_window_attention(Q, K, V, window_size=w, n_sink=n_sink)

print("=== Attention Pattern Comparison (seq=32, window=8, sinks=4) ===")
for name, weights in [("Full Causal", w_full), ("Sliding Window", w_sw), ("Sink + Window", w_sink)]:
    nz = (weights[0, 0] > 1e-6).sum().item()
    print(f"  {name:<16s}: {nz:>4d} non-zero entries")

# Show which positions the last token attends to
print(f"\nLast token (q={S-1}) attends to:")
for name, weights in [("Full Causal", w_full), ("Sliding Window", w_sw), ("Sink + Window", w_sink)]:
    attended = (weights[0, 0, -1] > 1e-6).nonzero(as_tuple=True)[0].tolist()
    print(f"  {name:<16s}: positions {attended}")
print(f"\nSink + Window preserves initial tokens [0..{n_sink-1}] + window [{S-w}..{S-1}]")

=== Attention Pattern Comparison (seq=32, window=8, sinks=4) ===
  Full Causal     :  528 non-zero entries
  Sliding Window  :  228 non-zero entries
  Sink + Window   :  318 non-zero entries

Last token (q=31) attends to:
  Full Causal     : positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
  Sliding Window  : positions [24, 25, 26, 27, 28, 29, 30, 31]
  Sink + Window   : positions [0, 1, 2, 3, 24, 25, 26, 27, 28, 29, 30, 31]

Sink + Window preserves initial tokens [0..3] + window [24..31]


In [6]:
# Visualize the three attention masks as ASCII art

def visualize_mask(weights, title, size=24):
    """Print an ASCII heatmap of attention weights."""
    w = weights[0, 0, :size, :size].numpy()
    print(f"\n--- {title} ---")
    print("     " + "".join(f"{j:>2d}" for j in range(size)))
    for i in range(size):
        row = f"q{i:>2d}: "
        for j in range(size):
            if w[i, j] > 0.1:
                row += " █"
            elif w[i, j] > 0.01:
                row += " ▒"
            elif w[i, j] > 1e-6:
                row += " ░"
            else:
                row += " ."
        print(row)

# Recompute with more visible seq_len
S2 = 24
Q2 = torch.randn(1, 1, S2, 16)
K2 = torch.randn(1, 1, S2, 16)
V2 = torch.randn(1, 1, S2, 16)

_, w2_full = full_causal_attention(Q2, K2, V2)
_, w2_sw = sliding_window_attention(Q2, K2, V2, window_size=6)
_, w2_sink = sink_window_attention(Q2, K2, V2, window_size=6, n_sink=3)

visualize_mask(w2_full, "Full Causal Attention", S2)
visualize_mask(w2_sw, "Sliding Window (w=6)", S2)
visualize_mask(w2_sink, "Sink (3) + Window (6)", S2)


--- Full Causal Attention ---
      0 1 2 3 4 5 6 7 8 91011121314151617181920212223
q 0:  █ . . . . . . . . . . . . . . . . . . . . . . .
q 1:  █ █ . . . . . . . . . . . . . . . . . . . . . .
q 2:  █ █ █ . . . . . . . . . . . . . . . . . . . . .
q 3:  █ ▒ █ █ . . . . . . . . . . . . . . . . . . . .
q 4:  █ █ ▒ █ █ . . . . . . . . . . . . . . . . . . .
q 5:  ▒ █ ▒ █ █ ▒ . . . . . . . . . . . . . . . . . .
q 6:  █ ▒ █ ▒ ▒ █ ▒ . . . . . . . . . . . . . . . . .
q 7:  ▒ ▒ █ ▒ █ █ █ ▒ . . . . . . . . . . . . . . . .
q 8:  ▒ █ ▒ ▒ █ ▒ █ ▒ █ . . . . . . . . . . . . . . .
q 9:  ▒ ▒ ▒ ▒ ▒ █ ▒ █ █ █ . . . . . . . . . . . . . .
q10:  █ █ ▒ █ ▒ █ ▒ ▒ ▒ ░ █ . . . . . . . . . . . . .
q11:  ▒ ▒ ▒ ▒ █ ▒ ▒ █ ▒ █ ░ ░ . . . . . . . . . . . .
q12:  █ ▒ ▒ █ █ █ ▒ ▒ ▒ ▒ ▒ █ ░ . . . . . . . . . . .
q13:  ▒ █ ▒ ▒ █ ▒ █ ▒ ▒ ▒ ▒ ▒ ▒ ▒ . . . . . . . . . .
q14:  ░ █ ▒ ░ ▒ ▒ ▒ ░ ▒ █ ▒ ▒ █ ▒ ▒ . . . . . . . . .
q15:  ▒ █ ▒ █ █ ▒ ▒ ▒ ▒ ▒ ░ ▒ ▒ █ ▒ ▒ . . . . . . . .
q16:  ▒ ▒ ▒ ░ █ ░ █ █ █ ░ ▒ ▒ ▒ ▒ ▒ ▒ ▒ . . . . . .

## 5. Dynamic Sparse Attention

Static patterns are simple but may miss important tokens outside the window. **Dynamic methods** select which keys to attend to based on the actual query content.

### MInference (Microsoft, 2024)
Identified three dominant sparse patterns in long-context LLMs:
1. **A-shape (Vertical-Slash)**: Initial column + recent diagonal — similar to attention sinks + local
2. **Vertical-Slash**: Vertical lines at specific positions (punctuation, separators)
3. **Block-Sparse**: Dense blocks on diagonal with scattered important blocks

### Quest (2024)
KV cache is divided into **pages** of size $p$. For each page, compute a representative score using `max(K_page)` and `min(K_page)` per channel, then score pages against the query. Only load top-scoring pages for attention.

$$\text{score}_j = Q \cdot K_{\text{page}_j}^{\text{max}} + Q \cdot K_{\text{page}_j}^{\text{min}}$$

Let's implement a simplified Quest-style page scoring.

In [7]:
def quest_page_attention(Q, K, V, page_size=8, top_pages=4):
    """
    Simplified Quest-style paged attention.
    1. Divide K,V into pages of size `page_size`
    2. Score each page using max/min key statistics
    3. Select top pages, only compute attention on those

    Args:
        Q: (batch, heads, 1, head_dim)  — single query (decode step)
        K: (batch, heads, seq_len, head_dim)
        V: (batch, heads, seq_len, head_dim)
    Returns:
        output, selected_page_indices
    """
    B, H, S, D = K.shape
    n_pages = (S + page_size - 1) // page_size
    scale = 1.0 / math.sqrt(D)

    # Pad K to even page boundary
    pad_len = n_pages * page_size - S
    if pad_len > 0:
        K_padded = F.pad(K, (0, 0, 0, pad_len), value=0)
        V_padded = F.pad(V, (0, 0, 0, pad_len), value=0)
    else:
        K_padded = K
        V_padded = V

    # Reshape into pages: (B, H, n_pages, page_size, D)
    K_pages = K_padded.view(B, H, n_pages, page_size, D)
    V_pages = V_padded.view(B, H, n_pages, page_size, D)

    # Compute page statistics
    K_max = K_pages.max(dim=3).values   # (B, H, n_pages, D)
    K_min = K_pages.min(dim=3).values   # (B, H, n_pages, D)

    # Score each page: Q * K_max + Q * K_min
    q = Q.squeeze(2)  # (B, H, D)
    page_scores = (q.unsqueeze(2) * K_max).sum(-1) + (q.unsqueeze(2) * K_min).sum(-1)
    # (B, H, n_pages)

    # Select top pages
    top_k_pages = min(top_pages, n_pages)
    _, page_idx = torch.topk(page_scores, top_k_pages, dim=-1)  # (B, H, top_k_pages)

    # Gather selected pages
    page_idx_expanded = page_idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, -1, page_size, D)
    K_selected = K_pages.gather(2, page_idx_expanded)  # (B, H, top_k, page_size, D)
    V_selected = V_pages.gather(2, page_idx_expanded)

    # Flatten selected pages
    sel_len = top_k_pages * page_size
    K_sel = K_selected.reshape(B, H, sel_len, D)
    V_sel = V_selected.reshape(B, H, sel_len, D)

    # Standard attention on selected subset
    scores = torch.matmul(Q, K_sel.transpose(-2, -1)) * scale
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V_sel)

    return output, page_idx


# Demo
B, H, S, D = 1, 2, 64, 32
Q_dec = torch.randn(B, H, 1, D)  # single decode query
K_all = torch.randn(B, H, S, D)
V_all = torch.randn(B, H, S, D)

page_size = 8
top_pages = 4

out_quest, selected = quest_page_attention(Q_dec, K_all, V_all,
                                           page_size=page_size, top_pages=top_pages)

# Full attention for comparison
scale = 1.0 / math.sqrt(D)
full_scores = torch.matmul(Q_dec, K_all.transpose(-2, -1)) * scale
full_weights = torch.softmax(full_scores, dim=-1)
out_full_dec = torch.matmul(full_weights, V_all)

n_pages = S // page_size
print("=== Quest Page Attention ===")
print(f"Total pages: {n_pages}, Selected: {top_pages}")
print(f"Full attention: {S} KV entries processed")
print(f"Quest attention: {top_pages * page_size} KV entries processed ({top_pages*page_size/S*100:.0f}%)")
print(f"Selected page indices (head 0): {selected[0, 0].tolist()}")
print(f"Output MSE vs full: {F.mse_loss(out_quest, out_full_dec).item():.6f}")

=== Quest Page Attention ===
Total pages: 8, Selected: 4
Full attention: 64 KV entries processed
Quest attention: 32 KV entries processed (50%)
Selected page indices (head 0): [4, 3, 6, 0]
Output MSE vs full: 0.023043


## 6. PagedAttention — Memory Management

In serving scenarios, the KV-Cache for different requests has different lengths. Naive pre-allocation wastes memory. **PagedAttention** (vLLM, Kwon et al., 2023) borrows the concept of virtual memory paging from operating systems:

| Concept | OS Virtual Memory | PagedAttention |
|---------|-------------------|----------------|
| Block/Page | Fixed-size memory page | Fixed-size KV block |
| Page Table | Maps virtual → physical address | Maps logical KV position → physical block |
| Fragmentation | Internal fragmentation < 1 page | Waste < 1 KV block |

**Benefits**:
- Eliminates pre-allocation waste (2-4× more requests in same memory)
- Enables copy-on-write for beam search (shared prefix)
- Efficient batching of variable-length sequences

Let's simulate the core idea: block-based KV-Cache management.

In [8]:
class PagedKVCache:
    """
    Simplified PagedAttention KV-Cache manager.

    Instead of allocating a contiguous (max_seq_len, head_dim) buffer per request,
    we allocate fixed-size blocks from a shared pool.
    """
    def __init__(self, num_blocks, block_size, num_heads, head_dim):
        self.block_size = block_size
        self.num_blocks = num_blocks
        # Shared physical memory pool
        self.k_pool = torch.zeros(num_blocks, block_size, num_heads, head_dim)
        self.v_pool = torch.zeros(num_blocks, block_size, num_heads, head_dim)
        # Track which blocks are free
        self.free_blocks = list(range(num_blocks))
        # Page table per request: request_id -> [block_ids]
        self.page_tables = {}
        # Current fill position per request
        self.positions = {}

    def allocate(self, request_id):
        """Register a new request."""
        self.page_tables[request_id] = []
        self.positions[request_id] = 0

    def _ensure_space(self, request_id):
        """Allocate a new block if current one is full."""
        pos = self.positions[request_id]
        blocks = self.page_tables[request_id]
        needed_blocks = (pos // self.block_size) + 1
        while len(blocks) < needed_blocks:
            if not self.free_blocks:
                raise RuntimeError("Out of KV cache blocks!")
            blocks.append(self.free_blocks.pop())

    def append(self, request_id, k, v):
        """
        Append a new KV entry (single token) to the request's cache.
        k, v: (num_heads, head_dim)
        """
        self._ensure_space(request_id)
        pos = self.positions[request_id]
        block_idx = pos // self.block_size
        offset = pos % self.block_size
        physical_block = self.page_tables[request_id][block_idx]
        self.k_pool[physical_block, offset] = k
        self.v_pool[physical_block, offset] = v
        self.positions[request_id] = pos + 1

    def get_kv(self, request_id):
        """Retrieve the full K,V cache for a request (scatter-gathered from blocks)."""
        pos = self.positions[request_id]
        blocks = self.page_tables[request_id]
        k_list, v_list = [], []
        remaining = pos
        for bid in blocks:
            n = min(remaining, self.block_size)
            k_list.append(self.k_pool[bid, :n])
            v_list.append(self.v_pool[bid, :n])
            remaining -= n
        return torch.cat(k_list, dim=0), torch.cat(v_list, dim=0)  # (seq_len, heads, dim)

    def free(self, request_id):
        """Release all blocks for a completed request."""
        self.free_blocks.extend(self.page_tables[request_id])
        del self.page_tables[request_id]
        del self.positions[request_id]

    def status(self):
        used = self.num_blocks - len(self.free_blocks)
        print(f"Blocks: {used}/{self.num_blocks} used, {len(self.free_blocks)} free")
        for rid, blocks in self.page_tables.items():
            pos = self.positions[rid]
            waste = len(blocks) * self.block_size - pos
            print(f"  Request {rid}: {pos} tokens, {len(blocks)} blocks, {waste} slots wasted")


# Simulate a serving scenario
cache = PagedKVCache(num_blocks=16, block_size=4, num_heads=2, head_dim=8)

print("=== PagedKVCache Simulation ===")
cache.allocate("req_A")
cache.allocate("req_B")

# Simulate token generation
for i in range(7):  # Request A: 7 tokens
    k = torch.randn(2, 8)
    v = torch.randn(2, 8)
    cache.append("req_A", k, v)

for i in range(3):  # Request B: 3 tokens
    k = torch.randn(2, 8)
    v = torch.randn(2, 8)
    cache.append("req_B", k, v)

cache.status()

# Retrieve KV for attention
k_a, v_a = cache.get_kv("req_A")
print(f"\nRetrieved K shape for req_A: {k_a.shape}")

# Free request A
cache.free("req_A")
print("\nAfter freeing req_A:")
cache.status()

# Waste analysis (compared to naive allocation)
max_ctx = 64
print(f"\nNaive allocation for 2 requests with max_ctx={max_ctx}: {2*max_ctx*2*8*2} bytes")
print(f"Paged allocation (actual usage): {(7+3)*2*8*2} bytes + minor block metadata")

=== PagedKVCache Simulation ===
Blocks: 3/16 used, 13 free
  Request req_A: 7 tokens, 2 blocks, 1 slots wasted
  Request req_B: 3 tokens, 1 blocks, 1 slots wasted

Retrieved K shape for req_A: torch.Size([7, 2, 8])

After freeing req_A:
Blocks: 1/16 used, 15 free
  Request req_B: 3 tokens, 1 blocks, 1 slots wasted

Naive allocation for 2 requests with max_ctx=64: 4096 bytes
Paged allocation (actual usage): 320 bytes + minor block metadata


## 7. Sparse Attention Comparison

Let's measure the practical speedup of sparse attention vs. full attention.

In [9]:
import time

def benchmark_attention(func, Q, K, V, n_trials=20, **kwargs):
    """Benchmark attention function, return avg time in ms."""
    # Warmup
    for _ in range(3):
        func(Q, K, V, **kwargs)
    times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        func(Q, K, V, **kwargs)
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000


print("=== Sparse Attention Benchmark (CPU) ===")
print(f"{'Seq Length':>10s} | {'Full':>10s} | {'Window(32)':>10s} | {'Sink+Win':>10s} | {'Speedup':>8s}")
print("-" * 60)

for s in [64, 128, 256, 512]:
    Q = torch.randn(1, 4, s, 32)
    K = torch.randn(1, 4, s, 32)
    V = torch.randn(1, 4, s, 32)

    t_full = benchmark_attention(full_causal_attention, Q, K, V)
    t_sw = benchmark_attention(sliding_window_attention, Q, K, V, window_size=32)
    t_sink = benchmark_attention(sink_window_attention, Q, K, V, window_size=32, n_sink=4)

    speedup = t_full / t_sw if t_sw > 0 else float('inf')
    print(f"{s:>10d} | {t_full:>8.2f}ms | {t_sw:>8.2f}ms | {t_sink:>8.2f}ms | {speedup:>7.2f}×")

print("\nNote: Our Python implementation uses dense mask + softmax, so speedup is modest.")
print("FlashAttention targets I/O-efficient dense (incl. causal) attention—not arbitrary sparse graphs. Custom sparse kernels (e.g. Triton) can skip unselected pairs; here we use dense tensors + masks, so speedups stay modest.")

=== Sparse Attention Benchmark (CPU) ===
Seq Length |       Full | Window(32) |   Sink+Win |  Speedup
------------------------------------------------------------
        64 |     0.08ms |     0.09ms |     0.09ms |    0.92×
       128 |     0.16ms |     0.17ms |     0.18ms |    0.99×
       256 |     0.23ms |     0.29ms |     0.29ms |    0.81×
       512 |     0.55ms |     0.61ms |     3.64ms |    0.90×

Note: Our Python implementation uses dense mask + softmax, so speedup is modest.
FlashAttention targets I/O-efficient dense (incl. causal) attention—not arbitrary sparse graphs. Custom sparse kernels (e.g. Triton) can skip unselected pairs; here we use dense tensors + masks, so speedups stay modest.


## Conclusions

### Technical Concepts Learned
- **Attention Sparsity**: Full dense attention costs $O(s^2)$; sparse patterns (window, sinks, blocks) cut compute/memory while the main motivation is **scaling**, not softmax entries being "mostly zero"
- **Sliding Window Attention**: Each query attends to $w$ recent tokens — $O(sw)$ complexity, used in Mistral/Mixtral
- **Attention Sinks (StreamingLLM)**: First few tokens receive disproportionate attention; keeping them stabilizes streaming inference with fixed KV-Cache
- **Dynamic Sparse Patterns**: MInference identifies 3 patterns (A-shape, vertical-slash, block-sparse); Quest scores KV pages to select top-scoring subsets
- **PagedAttention**: Block-based KV memory management eliminates fragmentation and enables efficient batched serving (vLLM)

### Experiment Further
- Run the sliding window attention on a real model and measure perplexity degradation vs. window size
- Implement copy-on-write in PagedKVCache for beam search (shared prefix)
- Try different page sizes in Quest and observe accuracy vs. speed tradeoff
- Measure actual memory savings with PagedAttention under concurrent serving load
- Explore training-based sparse methods like NSA (gated fusion of compression + selection + sliding window)

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
